In [1]:
import os
import sqlite3
from pathlib import Path
from typing import List, Dict, Any

from langchain_core.documents import Document
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader

print("SQL Database utilities and Document loaders imported successfully!")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_31616\2603430115.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase
d:\Code\AI\rag-mastery\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SQL Database utilities and Document loaders imported successfully!


In [2]:
from pathlib import Path
data_dir = Path('data') if Path('data').exists() else Path('../../data')
db_file = (data_dir / 'databases' / 'company.db').resolve()

# التأكد من وجود قاعدة البيانات التي تم تغذيتها مركزياً
print(f"[+] SQLite Database located at: {db_file} (Exists: {db_file.exists()})")


[+] SQLite Database located at: D:\Code\AI\rag-mastery\data\databases\company.db (Exists: True)


In [4]:
db_uri = f"sqlite:///{db_file.as_posix()}"
db = SQLDatabase.from_uri(db_uri)

print(f"[+] Usable Tables: {db.get_usable_table_names()}")
print("\n--- Employees Table Schema & Sample Records ---")
print(db.get_table_info(["employees"]))

print("--- Projects Table Schema & Sample Records ---")
print(db.get_table_info(["projects"]))

[+] Usable Tables: ['employees', 'projects']

--- Employees Table Schema & Sample Records ---

CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	role TEXT, 
	department TEXT, 
	salary REAL, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	role	department	salary
1	John Doe	Senior Developer	Engineering	95000.0
2	Jane Smith	Lead Data Scientist	Data Science	115000.0
3	Mike Johnson	DevOps Architect	Infrastructure	105000.0
*/
--- Projects Table Schema & Sample Records ---

CREATE TABLE projects (
	id INTEGER, 
	name TEXT, 
	status TEXT, 
	budget REAL, 
	lead_id INTEGER, 
	PRIMARY KEY (id), 
	FOREIGN KEY(lead_id) REFERENCES employees (id)
)

/*
3 rows from projects table:
id	name	status	budget	lead_id
1	RAG Knowledge Base	In Progress	50000.0	1
2	Customer Support AI Agent	Completed	75000.0	2
3	Kubernetes Cloud Migration	In Progress	120000.0	3
*/


In [5]:
def sql_to_documents(db_path: str) -> List[Document]:
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    documents: List[Document] = []

    # 1. استخراج أسماء الجداول من sqlite_master
    cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [t[0] for t in cur.fetchall()]

    for table_name in tables:
        # 2. استخراج بيانات الأعمدة عبر PRAGMA table_info
        cur.execute(f"PRAGMA table_info({table_name});")
        col_info = cur.fetchall()
        columns = [c[1] for c in col_info]

        # 3. استخراج الصفوف
        cur.execute(f"SELECT * FROM {table_name};")
        rows = cur.fetchall()

        # صياغة محتوى وصفي للجدول
        content_lines = [
            f"Database Table: {table_name}",
            f"Defined Columns: {', '.join(columns)}",
            f"Total Rows: {len(rows)}",
            "Sample Data:"
        ]
        for row in rows[:5]:
            row_str = " | ".join([f"{col}: {val}" for col, val in zip(columns, row)])
            content_lines.append(f"  - {row_str}")

        metadata = {
            "source": Path(db_path).name,
            "table_name": table_name,
            "row_count": len(rows),
            "columns_count": len(columns)
        }
        documents.append(Document(page_content="\n".join(content_lines), metadata=metadata))

    conn.close()
    return documents

table_docs = sql_to_documents(str(db_file))
print(f"[+] Generated {len(table_docs)} table schema documents.\n")
for doc in table_docs:
    print(f"=== Table: {doc.metadata['table_name']} ===")
    print(doc.page_content)
    print(f"Metadata: {doc.metadata}\n")

[+] Generated 2 table schema documents.

=== Table: employees ===
Database Table: employees
Defined Columns: id, name, role, department, salary
Total Rows: 4
Sample Data:
  - id: 1 | name: John Doe | role: Senior Developer | department: Engineering | salary: 95000.0
  - id: 2 | name: Jane Smith | role: Lead Data Scientist | department: Data Science | salary: 115000.0
  - id: 3 | name: Mike Johnson | role: DevOps Architect | department: Infrastructure | salary: 105000.0
  - id: 4 | name: Sarah Williams | role: Frontend Specialist | department: Engineering | salary: 85000.0
Metadata: {'source': 'company.db', 'table_name': 'employees', 'row_count': 4, 'columns_count': 5}

=== Table: projects ===
Database Table: projects
Defined Columns: id, name, status, budget, lead_id
Total Rows: 4
Sample Data:
  - id: 1 | name: RAG Knowledge Base | status: In Progress | budget: 50000.0 | lead_id: 1
  - id: 2 | name: Customer Support AI Agent | status: Completed | budget: 75000.0 | lead_id: 2
  - id: 3 